<a href="https://colab.research.google.com/github/AyushTayal777/Langchain/blob/main/13_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata

import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [4]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai faiss-cpu tiktoken python-dotenv

In [7]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_6044/1373140048.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [9]:
# STEP-1 - DATA INGESTION

video_id = "fZM3oX4xEyg"

try:
    ytt_api = YouTubeTranscriptApi()

    transcript = ytt_api.fetch(
        video_id,
        languages=["en"]
    )

    # Convert to plain text
    transcript_text = " ".join(
        snippet.text for snippet in transcript
    )

    print(transcript_text)

except TranscriptsDisabled:
    print("No captions available for this video.")

Hello all, my name is Krishna and welcome to my YouTube channel. So guys, I am super excited to start this new series on one of the most important technique which is right now being used in genative AI and agentic AI field that is nothing but rag. If you don't know the full form of rag, it is called as retrieval augmented generation. In this specific video, we will try to understand what exactly is rag. uh what are the disadvantages of just using the LLM model and how we are overcoming those disadvantages with the help of rag when should we use rag and what are the important pipelines that we should take a note while developing a rag application okay so all this topics we will be discussing and as we go ahead we are going to implement each and every important pipelines with the help of Jupyter notebook and I will also show you with the help of modular coding Right. So both the ways we will try to implement it. Now why I'm stressing on this specific series because nowadays every compani

In [10]:
# STEP-2 - TEXT SPLITTING
splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

docs = splitter.create_documents([transcript_text])

In [11]:
len(docs)

23

In [12]:
docs[0]

Document(metadata={}, page_content="Hello all, my name is Krishna and welcome to my YouTube channel. So guys, I am super excited to start this new series on one of the most important technique which is right now being used in genative AI and agentic AI field that is nothing but rag. If you don't know the full form of rag, it is called as retrieval augmented generation. In this specific video, we will try to understand what exactly is rag. uh what are the disadvantages of just using the LLM model and how we are overcoming those disadvantages with the help of rag when should we use rag and what are the important pipelines that we should take a note while developing a rag application okay so all this topics we will be discussing and as we go ahead we are going to implement each and every important pipelines with the help of Jupyter notebook and I will also show you with the help of modular coding Right. So both the ways we will try to implement it. Now why I'm stressing on this specific s

In [13]:
embeddings=GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')
vector_store= FAISS.from_documents(docs,embeddings)

In [14]:
vector_store.index_to_docstore_id

{0: 'a4c4a088-ebee-4741-9d3c-255322836604',
 1: '2c0eebfa-dfc0-4a10-9bb5-02f12b2bf630',
 2: '80482c5e-081f-4047-a2bb-fc442a92ae6a',
 3: '5d92cbb2-db1f-4407-b02e-08f25bfd60d9',
 4: '24d577bb-4f9a-4107-b4c9-61b58816bda2',
 5: '6a11193a-fac5-4bc2-9203-6265b525cd1e',
 6: '1f56db4d-9020-40e7-ac4d-f8d29b75b6fb',
 7: 'b918ac8b-0406-413e-8d3d-92e8cc055e10',
 8: '8e592ca8-3a21-409e-bb66-a73377ff4aca',
 9: 'ea0ea989-bd46-4588-ae42-3c343be417e7',
 10: '5948b7cc-a93e-424c-aa08-fb0af524ec00',
 11: 'd84a11af-69b2-4c8b-ba80-b790fa28b29e',
 12: 'ae75905f-21fd-48d9-883d-52e402b9868b',
 13: '02955b63-e9da-460d-83fc-d052fc81a5e1',
 14: '483676e6-9ce1-4241-adfd-a439a2ede5df',
 15: '0684923c-9a7c-4379-b3ae-f12277eec13f',
 16: '4eb4189b-9bad-49b8-a55e-3e11a73c4686',
 17: '6137ca5f-6ec3-4382-9424-76803e277ecc',
 18: '5b899018-e0d5-4f55-a956-4bc00a0b3d94',
 19: '64c0e5ee-e569-4712-9c2c-2493acadef83',
 20: '66b40edc-48a4-4829-b40a-9cbab62b9bdd',
 21: '8db74303-5fc6-48f4-a830-c18f31762941',
 22: '5cc90dcb-797f-

In [17]:
# STEP -3 - RETRIEVER
retriever = vector_store.as_retriever(search_types='similarity',search_kwargs={'k':4})

In [18]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78a5b5969e80>, search_kwargs={'k': 4})

In [19]:
retriever.invoke("what is rag")

[Document(id='80482c5e-081f-4047-a2bb-fc442a92ae6a', metadata={}, page_content="understand rag. Okay. So first of all let's go through the definition and then I will give you a brief idea what exactly rag is all about you know. So here you can clearly see that rag is the process of optimizing the output of a large language model. Okay. So it references an authorative knowledge base outside of it training data set source before get generating a response. LLMs are trained on vast volume of data as we all know and use billions of parameters to generally original output for task like question answering, translating and completing sentences. Rag extends the already powerful capabilities of LLM to specific domain or an organizational internal knowledge base all without the need to retrain the model. Okay. It is cost- effective approach to improve LLM output. So it's relevant, accurate and useful in various context. So this is just a basic definition. You can refer to this particular definiti

In [20]:
# STEP-4 PROMPT
prompt= PromptTemplate(
    template='you are a helpful assistant, answer only from the provided context, if the context is insufficient, just say you dont know. context: {context}, question: {question}',
    input_variables=['context','question']

)


In [21]:
question = 'is the topic of langgraph discussed in this video?'
result = retriever.invoke(question)

In [22]:
result

[Document(id='6137ca5f-6ec3-4382-9424-76803e277ecc', metadata={}, page_content="It is completely developed based on rag applications. Okay. Rag it is it is a kind of a rag application. In perplexity you have connected to various retrievers. You are connected to tools. You are connected to web search right and then it is summarizing the output and giving by the LLM. Right? and it also uses various LLMs itself. I'm also planning to mostly start a startup soon enough within a couple of weeks I guess and the kind of application that I'm developing is a rag application only and it solves a very good problem for a developer. Okay. So that is the reason I'm not being able to upload a lot of videos because I'm pretty much involved in those startups and working and developing a product that India can definitely remember. Okay. And this is how you know this is this is this is how things are and you can basically see how good uh you know the pipeline actually works and this is basically a traditi

In [24]:
context_text = "\n\n".join(doc.page_content for doc in result)
context_text

"It is completely developed based on rag applications. Okay. Rag it is it is a kind of a rag application. In perplexity you have connected to various retrievers. You are connected to tools. You are connected to web search right and then it is summarizing the output and giving by the LLM. Right? and it also uses various LLMs itself. I'm also planning to mostly start a startup soon enough within a couple of weeks I guess and the kind of application that I'm developing is a rag application only and it solves a very good problem for a developer. Okay. So that is the reason I'm not being able to upload a lot of videos because I'm pretty much involved in those startups and working and developing a product that India can definitely remember. Okay. And this is how you know this is this is this is how things are and you can basically see how good uh you know the pipeline actually works and this is basically a traditional rack. Now you may be thinking what all things we'll be discussing. Okay\n\

In [27]:
final_prompt = prompt.invoke({"context":context_text,"question":question})

In [28]:
final_prompt

StringPromptValue(text="you are a helpful assistant, answer only from the provided context, if the context is insufficient, just say you dont know. context: It is completely developed based on rag applications. Okay. Rag it is it is a kind of a rag application. In perplexity you have connected to various retrievers. You are connected to tools. You are connected to web search right and then it is summarizing the output and giving by the LLM. Right? and it also uses various LLMs itself. I'm also planning to mostly start a startup soon enough within a couple of weeks I guess and the kind of application that I'm developing is a rag application only and it solves a very good problem for a developer. Okay. So that is the reason I'm not being able to upload a lot of videos because I'm pretty much involved in those startups and working and developing a product that India can definitely remember. Okay. And this is how you know this is this is this is how things are and you can basically see how

In [29]:
# STEP - 5 - TEXT GENERATION

llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash')

In [30]:
answer = llm.invoke(final_prompt)
print(answer.text)

Based on the provided context, I don't know. (The topic of LangGraph is not mentioned in the text.)


In [31]:
# CHAIN METHOD

from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser



In [32]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [33]:
parallel_chain= RunnableParallel(
    {
        'context' :retriever | RunnableLambda(format_docs),
        'question':RunnablePassthrough()
    }
)

In [34]:
parallel_chain.invoke("what is rag")

{'context': "understand rag. Okay. So first of all let's go through the definition and then I will give you a brief idea what exactly rag is all about you know. So here you can clearly see that rag is the process of optimizing the output of a large language model. Okay. So it references an authorative knowledge base outside of it training data set source before get generating a response. LLMs are trained on vast volume of data as we all know and use billions of parameters to generally original output for task like question answering, translating and completing sentences. Rag extends the already powerful capabilities of LLM to specific domain or an organizational internal knowledge base all without the need to retrain the model. Okay. It is cost- effective approach to improve LLM output. So it's relevant, accurate and useful in various context. So this is just a basic definition. You can refer to this particular definition. So guys, now let's go ahead and understand about rag. So let's 

In [35]:
parser = StrOutputParser()

In [36]:
main_chain= parallel_chain | prompt | llm | parser

In [37]:
main_chain.invoke('can you summarize the video in 2 lines')

'This video explains the end-to-end pipeline of a traditional RAG (Retrieval-Augmented Generation) application, covering data parsing, embeddings, vector stores, and LLM context retrieval. \nThe speaker also discusses their upcoming RAG startup and plans to release comprehensive, long-form coding tutorials covering advanced topics like agentic RAG.'